<div dir="rtl">

# نوتبوك ٠٦ — قياس اختلافات أنماط الانتباه (فصحى ضد عامية)  ·  nb06: Probing Attention-Pattern Differences (MSA vs Masri)

النوتبوك ده بيجهّز تجربة مدتها ساعة واحدة باستخدام **TransformerLens** على موديل **Pythia-160m** لاستكشاف إزاي طبقات الموديل ورؤوس الانتباه (attention heads) بتتعامل مع جمل اللغة العربية الفصحى (MSA) مقارنة بالجمل العامية المصرية (Masri).

**سياق الهاردوير (Strix Halo):** التجربة دي متصممة تشتغل على كارت الشاشة المدمج AMD Strix Halo باستخدام تعريفات ROCm 6.4. الموديل حجمه صغير ومناسب جداً للتشغيل المباشر والسريع.

This notebook sets up a 1-hour experiment using **TransformerLens** on **Pythia-160m** to explore how the model's layers and attention heads process Modern Standard Arabic (MSA) compared to Egyptian Arabic (Masri).

**Substrate Context (Strix Halo):** Designed to run on the AMD Strix Halo iGPU using ROCm 6.4 nightly. The model size is highly tractable for fast, interactive execution.

---

### أهداف التجربة  ·  Experiment Goals:
1. تحميل الموديل باستخدام `HookedTransformer` مع تفعيل توافقية ROCm.
2. سحب الجمل المقارنة (MSA/Masri) من ملف البيانات المشترك.
3. استخراج مصفوفات الانتباه (attention patterns) عند كل طبقة ورأس.
4. حساب مقاييس المقارنة (مثل انتروبيا الانتباه Attention Entropy والتركيز على الـ BOS).
5. رصد الفروق الاحصائية ورسمها لفهم أين تفترق اللهجتان داخل الموديل.

</div>

<div dir="rtl">

## ١. تهيئة البيئة البرمجية والـ GPU  ·  1. Environment & GPU Setup

عشان نشغّل الموديل على Strix Halo (المعروف بـ gfx1151) بدون مشاكل، لازم نعرّف المتغير `HSA_OVERRIDE_GFX_VERSION=11.5.1` قبل استدعاء `torch`.

To run PyTorch on Strix Halo (gfx1151 architecture) without execution failures, we export `HSA_OVERRIDE_GFX_VERSION=11.5.1` before importing `torch`.

</div>

In [ ]:
import os
# تهيئة توافقية ROCm لـ Strix Halo
# os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.5.1" (commented out: native support)

import json
import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from transformer_lens import HookedTransformer

# التحقق من وجود كارت الشاشة المدمج وتحميل مكتبات ROCm
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(0)}")

<div dir="rtl">

## ٢. تحميل البيانات المشتركة والتحكم في القياس  ·  2. Load Dataset & Length Controls

بنقرا البيانات من `msa-masri-pairs-v1.json`. علشان نتفادى تأثير مشكلة الـ "Tokenizer Tax" (اختلاف عدد التوكنز بين الفصحى والعامية)، هنعمل truncate أو نتحقق من الأطوال عشان نضمن مقارنة عادلة.

We read pairs from `msa-masri-pairs-v1.json`. To avoid biases from the "Tokenizer Tax" (different token counts between MSA and Masri), we will track and control for token lengths.

</div>

In [ ]:
# المسار الافتراضي لملف البيانات داخل المستودع
DATA_PATH = "../../eval/prompts/msa-masri-pairs-v1.json"

# إذا لم يكن الملف في هذا المسار النسبي، نكتب المسار الكامل
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/home/yassermakram/code/fanous-llm-lens/eval/prompts/msa-masri-pairs-v1.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

prompts = dataset.get("prompts", dataset.get("pairs"))
print(f"Loaded {len(prompts)} triples from dataset.")
print("Sample:", json.dumps(prompts[0], indent=2, ensure_ascii=False))

<div dir="rtl">

## ٣. تحميل الموديل والتوكنايزر  ·  3. Model & Tokenizer Loading

هنحمل موديل **Pythia-160m** المفرّغ من التكرار (`pythia-160m-deduped`).

We load the **Pythia-160m-deduped** model using HookedTransformer.

</div>

In [ ]:
MODEL_NAME = "EleutherAI/pythia-160m-deduped"
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
print(f"Model layers: {model.cfg.n_layers}, Heads: {model.cfg.n_heads}, Dim: {model.cfg.d_model}")

<div dir="rtl">

## ٤. استخراج أنماط الانتباه  ·  4. Extracting Attention Patterns

هنكتب دالة لتشغيل الجملة وحفظ التنشيطات (caching). مصفوفة الانتباه اللي بنسحبها بيكون شكلها:
`pattern` ← `(batch, heads, seq_len, seq_len)`

We write a helper to run the prompt and extract attention pattern tensors of shape:
`pattern` ← `(batch_size, n_heads, seq_len, seq_len)`

</div>

In [ ]:
def get_attention_matrix(text):
    """
    Runs a forward pass and returns all attention patterns.
    Output shape per layer: (1, n_heads, seq_len, seq_len) <-- (b, h, s, s)
    """
    tokens = model.to_tokens(text)
    logits, cache = model.run_with_cache(tokens, remove_batch_dim=False)
    
    # استخراج مصفوفات الانتباه لكل الطبقات
    patterns = []
    for layer in range(model.cfg.n_layers):
        # cache["pattern", layer] has shape (1, n_heads, seq_len, seq_len)
        patterns.append(cache["pattern", layer].detach().cpu())
        
    return tokens.cpu(), patterns

<div dir="rtl">

## ٥. حساب مقاييس المقارنة  ·  5. Computing Metrics

هنقيس الفروقات باستخدام ثلاث مقاييس أساسية:
1. **انتروبيا الانتباه (Attention Entropy):** عشان نعرف هل انتباه الرأس مشتت (عالي) ولا مركز على توكن معين (قليل).
2. **الانتباه لأول توكن (BOS Attention):** مؤشر على عدم وجود شيء واضح ينتبه إليه الرأس (غالباً بينتبه للـ BOS كـ fallback).
3. **الانتباه الموضعي المباشر (Local Diagonal Attention):** قياس مدى انتباه الموديل للتوكن السابق مباشرة.

We compute three metrics on the attention patterns:
1. **Attention Entropy:** Indicates search dispersion vs sharp focus.
2. **BOS Attention:** Often acts as a "null attention" sink when no pattern matches.
3. **Local Diagonal Attention:** Measures attention focus on immediately preceding tokens.

</div>

In [ ]:
def compute_entropy(attn_matrix):
    """
    Input: (n_heads, seq_len, seq_len) <-- (h, s, s)
    Output: (n_heads, seq_len) <-- (h, s) representing entropy per head per token query
    """
    # لتفادي لوغاريتم الصفر
    eps = 1e-9
    entropy = -torch.sum(attn_matrix * torch.log(attn_matrix + eps), dim=-1)
    return entropy

def analyze_prompt_pair(msa_text, masri_text):
    msa_tokens, msa_pat = get_attention_matrix(msa_text)
    masri_tokens, masri_pat = get_attention_matrix(masri_text)
    
    results = []
    # نقارن الطبقات والرؤوس
    for layer in range(model.cfg.n_layers):
        # Shape: (n_heads, seq_len, seq_len)
        m_attn = msa_pat[layer][0]
        ma_attn = masri_pat[layer][0]
        
        # حساب الانتروبيا المتوسطة عبر الجملة لكل رأس
        m_ent = compute_entropy(m_attn).mean(dim=-1)     # Shape: (n_heads,)
        ma_ent = compute_entropy(ma_attn).mean(dim=-1)   # Shape: (n_heads,)
        
        # حساب الانتباه لـ BOS (التوكن رقم 0)
        m_bos = m_attn[:, :, 0].mean(dim=-1)             # Shape: (n_heads,)
        ma_bos = ma_attn[:, :, 0].mean(dim=-1)           # Shape: (n_heads,)
        
        for head in range(model.cfg.n_heads):
            results.append({
                "layer": layer,
                "head": head,
                "msa_entropy": m_ent[head].item(),
                "masri_entropy": ma_ent[head].item(),
                "msa_bos_attn": m_bos[head].item(),
                "masri_bos_attn": ma_bos[head].item(),
                "entropy_diff": ma_ent[head].item() - m_ent[head].item(),
                "bos_attn_diff": ma_bos[head].item() - m_bos[head].item()
            })
            
    return pd.DataFrame(results)

<div dir="rtl">

## ٦. تشغيل التحليل والمقارنة الاحصائية  ·  6. Running Probes & Statistical Comparison

دلوقتي هنشغّل التحليل على كل الأزواج المتاحة ونجمّع النتائج في DataFrame واحد عشان نرسم الفروق بين الطبقات.

Now we run the analysis over the whole dataset and aggregate the metrics to plot the differences across layers.

</div>

In [ ]:
all_dfs = []
for p in prompts:
    df_pair = analyze_prompt_pair(p["msa"], p["masri"])
    df_pair["prompt_id"] = p["id"]
    df_pair["category"] = p["category"]
    all_dfs.append(df_pair)

summary_df = pd.concat(all_dfs, ignore_index=True)

# حساب متوسط الفروقات عبر كل الجمل
grouped = summary_df.groupby(["layer", "head"]).mean(numeric_only=True).reset_index()
print(grouped.head(10))

<div dir="rtl">

## ٧. الرسم البياني للنتائج  ·  7. Visualizing Attention Divergence

هنرسم خريطة حرارية (heatmap) بتوضح الفروق في الـ Attention Entropy بين الفصحى والعامية عند كل رأس طبقة.

We plot a heatmap of differences in Attention Entropy (Masri - MSA) for each head at each layer.

</div>

In [ ]:
# تحويل شكل البيانات ليناسب الرسم
entropy_pivot = grouped.pivot(index="layer", columns="head", values="entropy_diff")

fig = px.imshow(
    entropy_pivot.values,
    labels=dict(x="Attention Head", y="Layer", color="Entropy Difference (Masri - MSA)"),
    x=entropy_pivot.columns,
    y=entropy_pivot.index,
    color_continuous_scale="RdBu",
    title="Attention Entropy Divergence (Masri vs MSA) across Pythia-160m"
)
fig.show()

<div dir="rtl">

### تفسير خريطة الحرارة وإنتروبيا الانتباه  ·  Heatmap & Attention Entropy Interpretation:

الرسم البياني ده بيوضّح **تباعد إنتروبيا الانتباه (Attention Entropy Divergence)** بين العامية المصرية والفصحى. الإنتروبيا هنا هو مقياس لمدى تشتت أو تركيز الموديل:
- **إنتروبيا عالي (حيرة وتشتت):** الموديل مش قادر يحدد توكن معين يركز عليه، والانتباه بيتوزع على كلمات كتير.
- **إنتروبيا قليل (تركيز حاد):** الموديل مركز انتباهه بشكل قوي على كلمة أو توكن محدد.

#### دلالات الألوان في الرسم البياني:

1. **المناطق الحمراء (فروقات موجبة - Masri Entropy > MSA):**
   - بتعبر عن رؤوس الانتباه اللي بتواجه **تشتت وحيرة أعلى** لما تقرأ عامية مصرية مقارنة بالفصحى.
   - دي بتمثل مواضع الاختلافات النحوية والصرفية اللي الموديل مش متعود عليها من بيانات التدريب (زي طبقة 9 رأس 7)، حيث ينتشر الانتباه بحثاً عن سياق مألوف.

2. **المناطق الزرقاء (فروقات سالبة - MSA Entropy > Masri):**
   - بتعبر عن رؤوس انتباه الإنتروبيا بتاعها **أقل (مركزة أكتر)** في العامية مقارنة بالفصحى.
   - بالتحليل المقارن، وجدنا إن التركيز ده مش ناتج عن فهم أعمق، ولكن بسبب ظاهرة **بالوعة الانتباه (Attention Sink)**. 
   - لما الموديل (تحديداً في الطبقات المتوسطة زي طبقة 4 رأس 4 ورأس 5) يقابل تراكيب عامية غريبة عليه، بيفشل في إيجاد روابط سياقية واضحة، فيقوم يرمي وزن الانتباه كله على توكن البداية (**BOS token** - التوكن رقم 0) كـ fallback افتراضي. وبما إن الانتباه كله بيروح لتوكن واحد، الإنتروبيا بتقل جداً وتظهر باللون الأزرق.

</div>

<div dir="rtl">

## ٨. الخلاصة والخطوات التالية  ·  8. Closing Recap & Next Handoff

**الملخص:**
- قسنا اختلافات مصفوفات الانتباه عند كل رأس وطبقة.
- رؤوس معينة هتظهر فروق واضحة في الانتروبيا، وده بيمثل مواضع تفرع معالجة اللهجات.
- الانتباه للـ BOS بيوضح لنا الرؤوس اللي بتـ"تسقط" (fall back) لما الموديل يقابل تراكيب لغوية عامية غير مألوفة.

**الخطوة الجاية (نوتبوك ٠٧):**
بعد ما حددنا الرؤوس الأكثر تأثراً باختلاف اللهجة، هنبني تجربة تعديل وتوجيه النشاط (Activation Steering) لمحاولة تقريب معالجة الموديل للجمل العامية لتشابه الفصحى وقياس تأثير ده على جودة المخرجات.

**Recap:**
- We measured differences in attention patterns across layers and heads.
- Certain heads will show sharp entropy deltas, marking dialectal branching points.
- BOS attention acts as a diagnostic of heads failing to resolve dialectal grammar.

**Next Handoff (nb07):**
With the sensitive dialectal heads mapped, we will design an **Activation Steering** experiment to guide Pythia's Masri representations towards MSA pathways and measure output improvement.

</div>